<a href="https://colab.research.google.com/github/obaidah3/rag-ecommerce-chatbot/blob/main/01_language_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Language Detection
Multi-class classifier (TF-IDF + Logistic Regression) on `papluca/language-identification`.

**Why TF-IDF + Logistic Regression instead of a deep model?**
- Language ID is a lexical/character-pattern problem (specific letters, n-grams, stopwords) — classical ML with **char n-grams** solves it very well and trains in seconds instead of hours.
- This lets you spend your limited time budget on the harder modules (RAG, sentiment).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/chatbot_project"
MODELS_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Models will be saved to:", MODELS_DIR)

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/chatbot_project/models


In [ ]:
!pip install -q datasets scikit-learn joblib

In [ ]:
from datasets import load_dataset

ds = load_dataset("papluca/language-identification")
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()


README.md:   0%|          | 0.00/4.99k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 12.0MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

(70000, 2) (10000, 2) (10000, 2)


,labels,text
0,pt,"os chefes de defesa da estónia, letónia, lituâ..."
1,bg,размерът на хоризонталната мрежа може да бъде ...
2,zh,很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把...
3,th,สำหรับ ของเก่า ที่ จริงจัง ลอง honeychurch ...
4,ru,Он увеличил давление .


## Preprocessing
For language ID, DON'T lowercase-strip too aggressively — case and diacritics can be signal.
We keep it light: strip extra whitespace only. The real signal comes from **character n-grams** (2-5 chars), which is why we use `analyzer='char_wb'` in TF-IDF instead of word n-grams.

In [ ]:
import re

def clean(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text"] = train_df["text"].apply(clean)
val_df["text"]   = val_df["text"].apply(clean)
test_df["text"]  = test_df["text"].apply(clean)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), max_features=50000)),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=-1))
])

pipe.fit(train_df["text"], train_df["labels"])


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char_wb', max_features=50000,
                                 ngram_range=(2, 5))),
                ('clf', LogisticRegression(max_iter=1000, n_jobs=-1))])

In [ ]:
val_preds = pipe.predict(val_df["text"])
print("Validation accuracy:", accuracy_score(val_df["labels"], val_preds))
print(classification_report(val_df["labels"], val_preds))


Validation accuracy: 0.9943
              precision    recall  f1-score   support

          ar       1.00      0.99      1.00       500
          bg       1.00      1.00      1.00       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       0.99      1.00      0.99       500
          es       1.00      1.00      1.00       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.95      0.98       500
          it       1.00      1.00      1.00       500
          ja       1.00      1.00      1.00       500
          nl       0.98      1.00      0.99       500
          pl       1.00      1.00      1.00       500
          pt       0.99      1.00      1.00       500
          ru       1.00      1.00      1.00       500
          sw       0.95      1.00      0.97       500
          th       1.00      1.00      1.00       500
          tr       0.99      1.00      1.00       500

In [ ]:
test_preds = pipe.predict(test_df["text"])
print("Test accuracy:", accuracy_score(test_df["labels"], test_preds))


Test accuracy: 0.995


In [ ]:
import joblib
joblib.dump(pipe, f"{MODELS_DIR}/language_detector.joblib")
print("saved model")

saved model


## Quick sanity check

In [ ]:
samples = ["Hello, where is my order?", "Bonjour, où est ma commande?", "مرحبا اين طلبي", "Hola, donde esta mi pedido"]
pipe.predict(samples)


array(['en', 'fr', 'ar', 'es'], dtype=object)